In [14]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt

from sklearn import linear_model
import lightgbm as lgbm

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold


In [15]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv',index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


In [16]:
print(X_train.head())
print(X_train.dtypes)
print(X_train.shape)

print(y_train.head())
print(y_train.describe())

print(X_train["ALLOCATION"].nunique())
print(X_train["GROUP"].nunique())
print(X_train["TS"].nunique())

               TS     ALLOCATION    RET_20    RET_19    RET_18    RET_17  \
ROW_ID                                                                     
0       DATE_0001  ALLOCATION_01 -0.018192 -0.000306 -0.006881 -0.002393   
1       DATE_0001  ALLOCATION_02 -0.006394 -0.001059  0.001565  0.000033   
2       DATE_0001  ALLOCATION_03 -0.016587 -0.004517 -0.005306  0.004314   
3       DATE_0001  ALLOCATION_04 -0.005344  0.002790  0.006937 -0.004246   
4       DATE_0001  ALLOCATION_05 -0.010506 -0.005491  0.007752 -0.012299   

          RET_16    RET_15    RET_14    RET_13  ...  SIGNED_VOLUME_8  \
ROW_ID                                          ...                    
0       0.000507 -0.001270 -0.002539  0.002830  ...         0.818730   
1       0.002829  0.001725  0.000875 -0.002160  ...        -1.390336   
2       0.006471 -0.005868 -0.005030 -0.001488  ...         0.961318   
3      -0.005051 -0.000330 -0.000117 -0.005209  ...        -0.483377   
4       0.002191  0.003282  0.00049

In [17]:
X_train["RET1_CS_MEAN"] = X_train.groupby("TS")["RET_1"].transform("mean")
X_train["RET1_CS_STD"] = X_train.groupby("TS")["RET_1"].transform("std")
X_train["RET1_CS_MEDIAN"] = X_train.groupby("TS")["RET_1"].transform("median")


In [18]:
print("Rows:", len(X_train))

print(
    "Unique allocations:",
    X_train["ALLOCATION"].nunique()
)

print(
    "Unique TS:",
    X_train["TS"].nunique()
)

print(
    "Rows per allocation:"
)
print(
    X_train.groupby("ALLOCATION").size().describe()
)

print(
    "Missingness:"
)
print(
    X_train.isna().mean().sort_values(ascending=False).head(10)
)
ret_cols = [f"RET_{i}" for i in range(1, 21)]

print(
    X_train.groupby("ALLOCATION")[ret_cols]
    .apply(lambda x: x.notna().all(axis=1).mean())
    .describe()
)

Rows: 527073
Unique allocations: 278
Unique TS: 2522
Rows per allocation:
count     278.000000
mean     1895.946043
std       388.833385
min        19.000000
25%      1431.000000
50%      1864.000000
75%      1931.000000
max      2439.000000
dtype: float64
Missingness:
SIGNED_VOLUME_1          0.735204
SIGNED_VOLUME_20         0.016123
MEDIAN_DAILY_TURNOVER    0.006902
SIGNED_VOLUME_19         0.005696
SIGNED_VOLUME_18         0.001630
SIGNED_VOLUME_17         0.000110
RET_20                   0.000110
RET_19                   0.000095
SIGNED_VOLUME_16         0.000095
SIGNED_VOLUME_15         0.000080
dtype: float64
count    278.000000
mean       0.995768
std        0.049014
min        0.421053
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
dtype: float64


In [19]:

ret_cols = [f"RET_{i}" for i in range(1, 21)]

# Create shifted signatures.
# For a row B that follows row A:
# B.RET_2 == A.RET_1
# B.RET_3 == A.RET_2
# ...
# B.RET_20 == A.RET_19

X = X_train.copy()

# Look at how often individual values repeat
for i in range(1, 6):
    a = f"RET_{i}"
    b = f"RET_{i+1}"

    print(
        a, "vs", b,
        "unique values:",
        X[a].nunique(),
        X[b].nunique()
    )

RET_1 vs RET_2 unique values: 525391 525363
RET_2 vs RET_3 unique values: 525363 525469
RET_3 vs RET_4 unique values: 525469 525418
RET_4 vs RET_5 unique values: 525418 525357
RET_5 vs RET_6 unique values: 525357 525501


In [20]:
print("Train allocations:", X_train["ALLOCATION"].nunique())
print("Test allocations:", X_test["ALLOCATION"].nunique())

train_alloc = set(X_train["ALLOCATION"])
test_alloc = set(X_test["ALLOCATION"])

print("Allocations in both:", len(train_alloc & test_alloc))

print("\nTrain TS:", X_train["TS"].nunique())
print("Test TS:", X_test["TS"].nunique())

train_ts = set(X_train["TS"])
test_ts = set(X_test["TS"])

print("TS in both:", len(train_ts & test_ts))

Train allocations: 278
Test allocations: 278
Allocations in both: 278

Train TS: 2522
Test TS: 120
TS in both: 0


In [21]:
import numpy as np
import pandas as pd

X = X_train

left = X[["ALLOCATION"] + [f"RET_{i}" for i in range(1, 20)]].copy()
right = X[["ALLOCATION"] + [f"RET_{i}" for i in range(2, 21)]].copy()

left.columns = ["ALLOCATION"] + [f"R{i}" for i in range(1, 20)]
right.columns = ["ALLOCATION"] + [f"R{i}" for i in range(1, 20)]

merged = left.merge(
    right.drop_duplicates(),
    on=["ALLOCATION"] + [f"R{i}" for i in range(1, 20)],
    how="inner"
)

print("Potential consecutive matches:", len(merged))

Potential consecutive matches: 0


In [22]:
left = X_train[["ALLOCATION", "RET_1"]].copy()
right = X_train[["ALLOCATION", "RET_2"]].copy()

right.columns = ["ALLOCATION", "RET_1"]

merged = left.merge(
    right.drop_duplicates(),
    on=["ALLOCATION", "RET_1"],
    how="inner"
)

print("1-day exact matches:", len(merged))

1-day exact matches: 16


In [23]:
left = X_train[["ALLOCATION", "RET_1"]].copy()
right = X_train[["ALLOCATION", "RET_2"]].copy()

left["RET_1"] = left["RET_1"].round(6)
right["RET_2"] = right["RET_2"].round(6)

right = right.rename(columns={"RET_2": "RET_1"})

merged = left.merge(
    right.drop_duplicates(),
    on=["ALLOCATION", "RET_1"],
    how="inner"
)

print("Matches after rounding to 6 decimals:", len(merged))

Matches after rounding to 6 decimals: 106605


In [24]:
cols_left = [f"RET_{i}" for i in range(1, 6)]
cols_right = [f"RET_{i}" for i in range(2, 7)]

left = X_train[["ALLOCATION"] + cols_left].copy()
right = X_train[["ALLOCATION"] + cols_right].copy()

for c in cols_left:
    left[c] = left[c].round(6)

for c in cols_right:
    right[c] = right[c].round(6)

right.columns = ["ALLOCATION"] + cols_left

merged = left.merge(
    right.drop_duplicates(),
    on=["ALLOCATION"] + cols_left,
    how="inner"
)

print("5-return matches:", len(merged))

5-return matches: 0


In [25]:
left = X_train[["ALLOCATION", "RET_1"]].copy()
right = X_train[["ALLOCATION", "RET_2"]].copy()

left["RET_1"] = left["RET_1"].round(6)
right["RET_2"] = right["RET_2"].round(6)

counts = right.groupby(["ALLOCATION", "RET_2"]).size()

matched = left.merge(
    counts.rename("count"),
    left_on=["ALLOCATION", "RET_1"],
    right_index=True,
    how="inner"
)

print("Matched rows:", len(matched))
print("\nCandidate count distribution:")
print(matched["count"].value_counts().head(10))

Matched rows: 106605

Candidate count distribution:
count
1    91690
2    13325
3     1447
4      131
5       10
6        2
Name: count, dtype: int64


In [26]:
left = X_train[["ALLOCATION", "RET_1", "RET_2"]].copy()
right = X_train[["ALLOCATION", "RET_2", "RET_3"]].copy()

for c in ["RET_1", "RET_2"]:
    left[c] = left[c].round(6)

for c in ["RET_2", "RET_3"]:
    right[c] = right[c].round(6)

right.columns = ["ALLOCATION", "RET_1", "RET_2"]

merged = left.merge(
    right.drop_duplicates(),
    on=["ALLOCATION", "RET_1", "RET_2"],
    how="inner"
)

print("2-return matches:", len(merged))

2-return matches: 21
